### 0. Setup

In [1]:
# Librerías y claves

import requests                      # para hacer las llamadas a la API
import pandas as pd                  # para manejar los datos en tablas
import os                            # para leer variables del entorno
from dotenv import load_dotenv       # para cargar las claves desde el archivo .env

# Cargamos las claves del archivo .env (así no quedan escritas en el notebook)
load_dotenv()

APP_ID = os.getenv("ADZUNA_APP_ID")
APP_KEY = os.getenv("ADZUNA_APP_KEY")

# Comprobamos que se han cargado bien (sin enseñar la clave entera por seguridad)
print("App ID cargado:", APP_ID)
print("App Key cargada:", "Sí" if APP_KEY else "No")

App ID cargado: 2e3c82f6
App Key cargada: Sí


### 1. Estructura de la API

In [ ]:
# Llamada de prueba

# URL base de la API de Adzuna para búsqueda de empleos
# Estructura: .../jobs/{pais}/search/{pagina}
pais = "gb"          # gb = Reino Unido (solo para la prueba)
pagina = 1

url = f"https://api.adzuna.com/v1/api/jobs/{pais}/search/{pagina}"

# Parámetros de la llamada
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 5,      # solo 5 ofertas para la prueba
    "what": "data"              # palabra de búsqueda de ejemplo
}

# Hacemos llamada
respuesta = requests.get(url, params=params)

# Comprobamos si funciona (200 = OK)
print("Código de estado:", respuesta.status_code)

# Convertimos la respuesta a formato JSON (dicc de Python)
datos = respuesta.json()

# Vemos qué "claves" principales trae la respuesta
print("Claves principales:", datos.keys())

Código de estado: 200
Claves principales: dict_keys(['count', 'results', 'mean', '__CLASS__'])


### 2. Exploración de una oferta de empleo

In [3]:
# 'results' es la lista de ofertas. Cogemos la primera [0] para examinarla
primera_oferta = datos["results"][0]

# Vemos qué campos (claves) trae una oferta
print("Campos de una oferta:")
for campo in primera_oferta.keys():
    print(" -", campo)

Campos de una oferta:
 - location
 - title
 - salary_is_predicted
 - description
 - longitude
 - created
 - contract_type
 - category
 - id
 - adref
 - latitude
 - redirect_url
 - salary_max
 - company
 - salary_min
 - __CLASS__


**Selección de variables**

Tras explorar la estructura de una oferta, nos quedamos con **todos los campos relevantes para el análisis** y descartamos los que no aportan valor.

**Variables:**
- `title` → puesto o rol.
- `category` → sector (tech / finanzas / retail), nuestro eje principal de análisis.
- `salary_min` y `salary_max` → métrica central del proyecto.
- `salary_is_predicted` → indica si el salario es real (0) o estimado (1); marca la calidad del dato.
- `location` → ubicación geográfica (país, región, ciudad).
- `company` → empresa que publica la oferta.
- `contract_type` → tipo de contrato.
- `created` → fecha de publicación (dimensión temporal).

**Variables descartadas:**
- `description` → texto largo del anuncio, fuera del alcance del proyecto.
- `latitude` y `longitude` → coordenadas exactas innecesarias, la geografía ya queda cubierta con `location`.
- `id`, `adref`, `redirect_url` → identificadores y enlaces técnicos sin valor analítico
- `__CLASS__` → campo interno de la API

Nota: algunos campos son anidados (`category`, `company`, `location` contienen subcampos), por lo que en la fase de recogida extraeremos el dato concreto de cada uno.

In [4]:
# Ver las categorías disponibles y sus tags
url_cat = "https://api.adzuna.com/v1/api/jobs/gb/categories"
params_cat = {"app_id": APP_ID, "app_key": APP_KEY}
cats = requests.get(url_cat, params=params_cat).json()

for c in cats["results"]:
    print(c["tag"], "→", c["label"])

accounting-finance-jobs → Accounting & Finance Jobs
it-jobs → IT Jobs
sales-jobs → Sales Jobs
customer-services-jobs → Customer Services Jobs
engineering-jobs → Engineering Jobs
hr-jobs → HR & Recruitment Jobs
healthcare-nursing-jobs → Healthcare & Nursing Jobs
hospitality-catering-jobs → Hospitality & Catering Jobs
pr-advertising-marketing-jobs → PR, Advertising & Marketing Jobs
logistics-warehouse-jobs → Logistics & Warehouse Jobs
teaching-jobs → Teaching Jobs
trade-construction-jobs → Trade & Construction Jobs
admin-jobs → Admin Jobs
legal-jobs → Legal Jobs
creative-design-jobs → Creative & Design Jobs
graduate-jobs → Graduate Jobs
retail-jobs → Retail Jobs
consultancy-jobs → Consultancy Jobs
manufacturing-jobs → Manufacturing Jobs
scientific-qa-jobs → Scientific & QA Jobs
social-work-jobs → Social work Jobs
travel-jobs → Travel Jobs
energy-oil-gas-jobs → Energy, Oil & Gas Jobs
property-jobs → Property Jobs
charity-voluntary-jobs → Charity & Voluntary Jobs
domestic-help-cleaning-job

### 3. Función para recogida de ofertas

In [5]:
import time  # para hacer pausas entre llamadas y no saturar la API

def recoger_ofertas(pais, categoria, n_paginas=8, por_pagina=50):
    """
    Recoge ofertas de un país y categoría concretos.
    Devuelve una lista de diccionarios "planos" (subcampos ya extraídos).
    """
    ofertas = []

    for pagina in range(1, n_paginas + 1):
        url = f"https://api.adzuna.com/v1/api/jobs/{pais}/search/{pagina}"
        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "results_per_page": por_pagina,
            "category": categoria
        }

        respuesta = requests.get(url, params=params)

        # Si la llamada falla, avisamos y paramos este país/categoría
        if respuesta.status_code != 200:
            print(f"Error {respuesta.status_code} en {pais}/{categoria} pág {pagina}")
            break

        resultados = respuesta.json()["results"]

        # Si una página viene vacía, no hay más ofertas  paramos 
        if len(resultados) == 0:
            break

        # Extraemos solo las variables que nos interesan (aplanando los anidados)
        for oferta in resultados:
            ofertas.append({
                "title": oferta.get("title"),
                "category": oferta.get("category", {}).get("label"),
                "salary_min": oferta.get("salary_min"),
                "salary_max": oferta.get("salary_max"),
                "salary_is_predicted": oferta.get("salary_is_predicted"),
                "location": oferta.get("location", {}).get("display_name"),
                "company": oferta.get("company", {}).get("display_name"),
                "contract_type": oferta.get("contract_type"),
                "created": oferta.get("created"),
                "pais": pais  # añadimos el país para identificarlo después
            })

        time.sleep(1)  # pausa de 1 segundo entre páginas (buena práctica)

    return ofertas

### 4. Recogida de datos

In [6]:
# Prueba de recogida para validar los datos 

prueba = recoger_ofertas("gb", "it-jobs", n_paginas=1)  # solo 1 página = 1 llamada

# Lo pasamos a tabla para verlo bien
df_prueba = pd.DataFrame(prueba)

print("Número de ofertas:", len(df_prueba))
df_prueba.head(10)   # vemos las primeras 10

Número de ofertas: 50


,title,category,salary_min,salary_max,salary_is_predicted,location,company,contract_type,created,pais
0,"Senior Quality Manager – Infrastructure, Capab...",IT Jobs,42460.12,42460.12,1,"The Hill, Millom",BAE Systems,None,2026-06-01T00:52:25Z,gb
1,UI and UX Manager (Edgewing),IT Jobs,55607.64,55607.64,1,"Deepcut, Camberley",BAE Systems,None,2026-06-01T00:52:28Z,gb
2,IT Strategy Specialist (Edgewing),IT Jobs,44809.20,44809.20,1,"Artington, Guildford",BAE Systems,None,2026-06-01T16:02:02Z,gb
3,Procurement Leader,IT Jobs,37874.68,37874.68,1,"Far Arnside, Carnforth",BAE Systems,None,2026-05-22T16:00:01Z,gb
4,Survey Instrument Coordinator,IT Jobs,53257.79,53257.79,1,"Devonport, Plymouth",Kier Group,None,2026-06-01T00:52:32Z,gb
5,Applications Specialist,IT Jobs,36790.91,36790.91,1,"Bermuda Park, Nuneaton",Hologic,None,2026-06-01T06:30:17Z,gb
6,Remote Game Tester - Earn Extra Income!,IT Jobs,44369.05,44369.05,1,"Swansea, Wales",Babki,None,2026-05-13T22:26:44Z,gb
7,LTQR Manager,IT Jobs,61641.28,61641.28,1,"Devonport, Plymouth",Kier Group,None,2026-05-09T01:05:34Z,gb
8,Student Game Tester,IT Jobs,42181.96,42181.96,1,"Folkestone, Kent",Babki,None,2026-05-08T17:32:19Z,gb
9,Remote Game Tester - Earn Extra Income!,IT Jobs,43477.98,43477.98,1,"Bristol, South West England",Babki,None,2026-05-13T22:26:50Z,gb


**Cosas a destacar:**

Antes de lanzar la recogida completa, se ha probado con Reino Unido, IT Jobs, 1 página para comprobar si los datos sirven.

**Resultado: los datos son válidos para el proyecto.** Se obtienen 50 ofertas por página, con las variables clave rellenas: `salary_min`, `salary_max`, `category`, `location`, `company`, `pais` y `created`.

**Observaciones y limitaciones detectadas:**

1. **Salarios estimados:** en la muestra, todos los salarios tienen `salary_is_predicted = 1`, es decir, son estimaciones de la API Adzuna y no el salario exacto de la oferta y hay que tenerlo en cuenta como limitación.

2. **`salary_min` = `salary_max`:** en los salarios estimados, el mínimo y el máximo coinciden. Por tanto, la dispersión salarial no se calculará *dentro* de cada oferta, sino *entre* ofertas (por sector y país).

3. **Clasificación de sector amplia:** algunas ofertas en "IT Jobs" no son estrictamente tecnológicas (p. ej. perfiles de gestión o calidad). La categoría de Adzuna agrupa de forma amplia, por lo que el sector es una aproximación.

**Decisión:** usamos **todos los salarios, incluidos los estimados**, para disponer de un volumen de datos suficiente e incluir el carácter estimado como limitación del análisis.

In [7]:
# Recogida completa (4 países × 4 sectores)

# Definimos qué vamos a recoger
paises = ["es", "gb", "fr", "de"]            # España, Reino Unido, Francia, Alemania
categorias = ["it-jobs", "accounting-finance-jobs", "retail-jobs", "logistics-warehouse-jobs"]  # tech, finanzas, retail

# Aquí iremos acumulando todas las ofertas
todas_las_ofertas = []

# Recorremos cada país y, dentro, cada categoría
for pais in paises:
    for categoria in categorias:
        print(f"Recogiendo: {pais} / {categoria} ...")
        ofertas = recoger_ofertas(pais, categoria, n_paginas=8)
        print(f"   → {len(ofertas)} ofertas")
        todas_las_ofertas.extend(ofertas)   # las añadimos al total

# Lo convertimos todo en una tabla
df = pd.DataFrame(todas_las_ofertas)

print("\n Recogida terminada")
print("Total de ofertas recogidas:", len(df))

Recogiendo: es / it-jobs ...
   → 400 ofertas
Recogiendo: es / accounting-finance-jobs ...
   → 400 ofertas
Recogiendo: es / retail-jobs ...
   → 101 ofertas
Recogiendo: es / logistics-warehouse-jobs ...
   → 400 ofertas
Recogiendo: gb / it-jobs ...
   → 400 ofertas
Recogiendo: gb / accounting-finance-jobs ...
   → 400 ofertas
Recogiendo: gb / retail-jobs ...
   → 400 ofertas
Recogiendo: gb / logistics-warehouse-jobs ...
   → 400 ofertas
Recogiendo: fr / it-jobs ...
   → 400 ofertas
Recogiendo: fr / accounting-finance-jobs ...
   → 400 ofertas
Recogiendo: fr / retail-jobs ...
   → 400 ofertas
Recogiendo: fr / logistics-warehouse-jobs ...
   → 400 ofertas
Recogiendo: de / it-jobs ...
   → 400 ofertas
Recogiendo: de / accounting-finance-jobs ...
   → 400 ofertas
Recogiendo: de / retail-jobs ...
   → 400 ofertas
Recogiendo: de / logistics-warehouse-jobs ...
   → 400 ofertas

 Recogida terminada
Total de ofertas recogidas: 6101


In [8]:
# Guardar los datos en CSV

ruta = "Data/ofertas_adzuna_data_raw.csv"
df.to_csv(ruta, index=False, encoding="utf-8")

print(f" Datos guardados en: {ruta}")
print(f"   {len(df)} filas, {df.shape[1]} columnas")

 Datos guardados en: Data/ofertas_adzuna_data_raw.csv
   6101 filas, 10 columnas
